In [14]:
from getpass import getpass

import pandas as pd
import requests

EIA_KEY = getpass("EIA API key: ")
START = "2024-01-01T00"
END = "2024-01-07T23"
EIA_BASE = "https://api.eia.gov/v2/electricity/rto"

def fetch_eia_metric(type_code: str, metric_name: str) -> pd.DataFrame:
    """Fetch one hourly CISO metric and return its rows."""
    rows = []
    offset = 0
    total = None

    while total is None or offset < total:
        params = [
            ("api_key", EIA_KEY),
            ("frequency", "hourly"),
            ("start", START),
            ("end", END),
            ("data[]", "value"),
            ("facets[respondent][]", "CISO"),
            ("facets[type][]", type_code),
            ("sort[0][column]", "period"),
            ("sort[0][direction]", "asc"),
            ("length", "5000"),
            ("offset", str(offset)),
        ]

        response = requests.get(
            f"{EIA_BASE}/region-data/data/",
            params=params,
            timeout=30,
        )
        if not response.ok:
            raise RuntimeError(f"EIA request failed: HTTP {response.status_code}")

        result = response.json()
        if "error" in result:
            raise RuntimeError(f"EIA API error: {result['error']}")

        data = result["response"]
        batch = data.get("data", [])
        rows.extend(batch)
        total = int(data["total"])
        offset += len(batch)

        if not batch:
            break

    frame = pd.DataFrame(rows)
    if not frame.empty:
        frame["value"] = pd.to_numeric(frame["value"], errors="coerce")
        frame["period"] = pd.to_datetime(frame["period"], utc=True)
        frame = frame[["period", "value"]].rename(columns={"value": metric_name})

    return frame

# Only the two EIA measurements we need for this first step.
demand = fetch_eia_metric("D", "demand")
demand_forecast = fetch_eia_metric("DF", "day_ahead_demand_forecast")

grid = (
    demand.merge(demand_forecast, on="period", how="outer")
    .set_index("period")
    .sort_index()
)

# Weather features: temperature, wind, and sunlight.
weather_response = requests.get(
    "https://archive-api.open-meteo.com/v1/archive",
    params={
        "latitude": 36.8,
        "longitude": -119.4,
        "start_date": "2024-01-01",
        "end_date": "2024-01-07",
        "hourly": "temperature_2m,wind_speed_10m,shortwave_radiation",
        "timezone": "UTC",
    },
    timeout=30,
)
if not weather_response.ok:
    raise RuntimeError(f"Weather request failed: HTTP {weather_response.status_code}")

weather_data = weather_response.json()["hourly"]
weather = pd.DataFrame(weather_data)
weather["period"] = pd.to_datetime(weather.pop("time"), utc=True)
weather = weather.set_index("period").sort_index()

# Predictor data, aligned by UTC hour.
dataset = grid.join(weather, how="outer")

print(dataset.shape)
display(dataset.head())

(168, 5)


,demand,day_ahead_demand_forecast,temperature_2m,wind_speed_10m,shortwave_radiation
period,,,,,
2024-01-01 00:00:00+00:00,23126,22350,11.8,6.2,60.0
2024-01-01 01:00:00+00:00,23165,23747,10.4,6.4,18.0
2024-01-01 02:00:00+00:00,24138,25790,10.0,7.2,0.0
2024-01-01 03:00:00+00:00,25999,25714,9.2,7.1,0.0
2024-01-01 04:00:00+00:00,25812,25039,8.8,3.6,0.0


In [15]:
from io import BytesIO

import pandas as pd
import requests

url = "https://www.eia.gov/electricity/gridmonitor/knownissues/xls/CISO.xlsx"

response = requests.get(url, timeout=60)
response.raise_for_status()

workbook = pd.ExcelFile(BytesIO(response.content))
print("Sheets:", workbook.sheet_names)

emissions = pd.read_excel(workbook, sheet_name="Published Hourly Data")
print("Columns:", emissions.columns.tolist())
display(emissions.head())

Sheets: ['Published Hourly Data', 'Published Daily Data', 'BA Full Names', 'Hourly Charts', 'Daily Charts', 'Known Data Issues', 'Notes', 'Imbalance Thresholds', 'Range Thresholds']
Columns: ['BA', 'UTC time', 'Local date', 'Hour', 'Local time', 'Time zone', 'Generation only?', 'Demand forecast', 'Demand', 'Net generation', 'Total interchange', 'Imputed demand', 'Imputed net generation', 'Imputed total interchange', 'Adjusted demand', 'Adjusted net generation', 'Adjusted total interchange', 'NG: COL', 'NG: NG', 'NG: NUC', 'NG: OIL', 'NG: GEO', 'NG: WAT', 'NG: PS', 'NG: SUN', 'NG: SNB', 'NG: WND', 'NG: WNB', 'NG: BAT', 'NG: OES', 'NG: UES', 'NG: OTH', 'NG: UNK', 'Imputed COL Gen', 'Imputed NG Gen', 'Imputed NUC Gen', 'Imputed OIL Gen', 'Imputed GEO Gen', 'Imputed WAT Gen', 'Imputed PS Gen', 'Imputed SUN Gen', 'Imputed SNB Gen', 'Imputed WND Gen', 'Imputed WNB Gen', 'Imputed BAT Gen', 'Imputed OES Gen', 'Imputed UES Gen', 'Imputed OTH Gen', 'Imputed UNK Gen', 'Adjusted COL Gen', 'Adjuste

,BA,UTC time,Local date,Hour,Local time,Time zone,Generation only?,Demand forecast,Demand,Net generation,...,CO2 Emissions: OIL,CO2 Emissions: Other,CO2 Emissions Generated,CO2 Emissions Imported,CO2 Emissions Exported,CO2 Emissions Consumed,Positive Generation,Consumed Electricity,CO2 Emissions Intensity for Generated Electricity,CO2 Emissions Intensity for Consumed Electricity
0,CISO,2015-07-01 08:00:00,2015-07-01,1,2015-07-01 01:00:00,Pacific,N,28653.0,31486.0,25789.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,CISO,2015-07-01 09:00:00,2015-07-01,2,2015-07-01 02:00:00,Pacific,N,26895.0,28989.0,23688.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,CISO,2015-07-01 10:00:00,2015-07-01,3,2015-07-01 03:00:00,Pacific,N,25725.0,27416.0,22107.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,CISO,2015-07-01 11:00:00,2015-07-01,4,2015-07-01 04:00:00,Pacific,N,25192.0,26388.0,20877.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,CISO,2015-07-01 12:00:00,2015-07-01,5,2015-07-01 05:00:00,Pacific,N,25416.0,25661.0,20262.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [18]:
target_col = "CO2 Emissions Intensity for Consumed Electricity"

co2 = emissions[["UTC time", target_col]].copy()
co2["UTC time"] = pd.to_datetime(co2["UTC time"], utc=True)
co2[target_col] = pd.to_numeric(co2[target_col], errors="coerce")

co2 = (
    co2.rename(columns={
        "UTC time": "timestamp",
        target_col: "carbon_intensity",
    })
    .dropna(subset=["carbon_intensity"])
    .drop_duplicates(subset=["timestamp"], keep="last")
    .set_index("timestamp")
    .sort_index()
)

print("Rows:", len(co2))
print("Date range:", co2.index.min(), "to", co2.index.max())
display(co2.head())

Rows: 72686
Date range: 2018-07-01 08:00:00+00:00 to 2026-10-27 04:00:00+00:00


,carbon_intensity
timestamp,
2018-07-01 08:00:00+00:00,0.351911
2018-07-01 09:00:00+00:00,0.300958
2018-07-01 10:00:00+00:00,0.299753
2018-07-01 11:00:00+00:00,0.295412
2018-07-01 12:00:00+00:00,0.307658


In [ ]:
import numpy as np
import pandas as pd

y = co2["carbon_intensity"].astype(float).sort_index()

# Reserve the latest 20% of timestamps for evaluation.
cutoff = y.index[int(len(y) * 0.8)]
test_origins = y.index[y.index >= cutoff]

# Keep only origins with a full 48 hours of actual data after them.
test_origins = test_origins[:-48]

results = []

for lead_hours in range(24, 49):
    target_times = test_origins + pd.to_timedelta(lead_hours, unit="h")
    prior_week_times = target_times - pd.Timedelta(hours=168)

    actual = y.reindex(target_times).to_numpy()
    predicted = y.reindex(prior_week_times).to_numpy()

    valid = np.isfinite(actual) & np.isfinite(predicted)
    mae = np.mean(np.abs(actual[valid] - predicted[valid]))

    results.append({
        "lead_hours": lead_hours,
        "samples": int(valid.sum()),
        "MAE": mae,
    })

baseline_scores = pd.DataFrame(results)
display(baseline_scores)
print("Average MAE, 24–48 hour leads:", baseline_scores["MAE"].mean())


"""
"Suppose a forecast starts at Monday, July 8 at 12:00 UTC:
- For a 24-hour lead, the target is Tuesday, July 9 at 12:00. The baseline predicts that hour’s intensity using the value from Tuesday, July 2 at 12:00—the same hour one week earlier.
- For a 48-hour lead, the target is Wednesday, July 10 at 12:00. It uses the value from Wednesday, July 3 at 12:00.
"""

,lead_hours,samples,MAE
0,24,14259,0.068143
1,25,14256,0.068146
2,26,14253,0.068138
3,27,14250,0.068128
4,28,14247,0.068116
5,29,14244,0.068111
6,30,14241,0.068099
7,31,14238,0.068081
8,32,14235,0.068070
9,33,14232,0.068068


Average MAE, 24–48 hour leads: 0.0680815759492579


In [22]:
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error

# Ensure lags mean actual hours, even if some timestamps are missing.
y = co2["carbon_intensity"].astype(float).sort_index()
hourly_index = pd.date_range(y.index.min(), y.index.max(), freq="h")
y = y.reindex(hourly_index)

# Each row uses information available at the forecast start time.
frame = pd.DataFrame(index=y.index)
frame["lag_1h"] = y.shift(1)
frame["lag_24h"] = y.shift(24)
frame["lag_168h"] = y.shift(168)

# Predict the carbon intensity 24 hours after each forecast start.
target_time = frame.index + pd.Timedelta(hours=24)
frame["target_hour"] = target_time.hour
frame["target_weekday"] = target_time.dayofweek
frame["target_month"] = target_time.month
frame["target"] = y.shift(-24)

# Same-hour-last-week benchmark for that 24-hour forecast.
frame["baseline"] = y.shift(144)

features = [
    "lag_1h", "lag_24h", "lag_168h",
    "target_hour", "target_weekday", "target_month",
]

# Chronological split; training labels stay before the test period.
cutoff = y.index[int(len(y) * 0.8)]
train = frame.loc[frame.index < cutoff - pd.Timedelta(hours=24)]
test = frame.loc[frame.index >= cutoff]

train = train.dropna(subset=features + ["target"])
test = test.dropna(subset=features + ["target", "baseline"])

model = HistGradientBoostingRegressor(
    max_iter=150,
    learning_rate=0.08,
    max_leaf_nodes=15,
    l2_regularization=1.0,
    random_state=42,
)
model.fit(train[features], train["target"])

model_mae = mean_absolute_error(test["target"], model.predict(test[features]))
baseline_mae = mean_absolute_error(test["target"], test["baseline"])

print(f"Training rows: {len(train):,}")
print(f"Test rows: {len(test):,}")
print(f"24-hour baseline MAE: {baseline_mae:.6f}")
print(f"24-hour model MAE:    {model_mae:.6f}")

Training rows: 57,795
Test rows: 14,027
24-hour baseline MAE: 0.068202
24-hour model MAE:    0.051552


In [23]:
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error

y = co2["carbon_intensity"].astype(float).sort_index()

# Make each row represent one actual UTC hour.
hourly_index = pd.date_range(y.index.min(), y.index.max(), freq="h")
y = y.reindex(hourly_index)

frame = pd.DataFrame(index=y.index)

# Inputs: past carbon intensity only.
frame["lag_72h"] = y.shift(72)
frame["lag_96h"] = y.shift(96)
frame["lag_240h"] = y.shift(240)

# Value to predict: carbon intensity 24 hours after forecast start.
frame["target"] = y.shift(-24)

# Baseline: same target hour from one week earlier.
frame["baseline"] = y.shift(144)

features = ["lag_72h", "lag_96h", "lag_240h"]

cutoff = y.index[int(len(y) * 0.8)]
train = frame.loc[frame.index < cutoff - pd.Timedelta(hours=24)]
test = frame.loc[frame.index >= cutoff]

train = train.dropna(subset=features + ["target"])
test = test.dropna(subset=features + ["target", "baseline"])

model = HistGradientBoostingRegressor(
    max_iter=150,
    learning_rate=0.08,
    max_leaf_nodes=15,
    l2_regularization=1.0,
    random_state=42,
)
model.fit(train[features], train["target"])

model_mae = mean_absolute_error(test["target"], model.predict(test[features]))
baseline_mae = mean_absolute_error(test["target"], test["baseline"])

print(f"Training rows: {len(train):,}")
print(f"Test rows: {len(test):,}")
print(f"24-hour baseline MAE: {baseline_mae:.6f}")
print(f"24-hour model MAE:    {model_mae:.6f}")

Training rows: 57,674
Test rows: 13,882
24-hour baseline MAE: 0.068037
24-hour model MAE:    0.068119


In [25]:
from getpass import getpass

import pandas as pd
import requests

eia_key = getpass("EIA API key: ")
api_start = "2019-01-01T00"
api_end = co2.index.max().strftime("%Y-%m-%dT%H")

url = "https://api.eia.gov/v2/electricity/rto/region-data/data"
rows = []
offset = 0
total = None

while total is None or offset < total:
    params = [
        ("api_key", eia_key),
        ("frequency", "hourly"),
        ("start", api_start),
        ("end", api_end),
        ("data[]", "value"),
        ("facets[respondent][]", "CISO"),
        ("facets[type][]", "DF"),  # Day-ahead demand forecast
        ("sort[0][column]", "period"),
        ("sort[0][direction]", "asc"),
        ("length", "5000"),
        ("offset", str(offset)),
    ]

    response = requests.get(url, params=params, timeout=30)
    if not response.ok:
        raise RuntimeError(f"EIA request failed: HTTP {response.status_code}")

    result = response.json()
    if "error" in result:
        raise RuntimeError(f"EIA API error: {result['error']}")

    data = result["response"]
    batch = data.get("data", [])
    rows.extend(batch)
    total = int(data["total"])
    offset += len(batch)

    if not batch:
        break

day_ahead = pd.DataFrame(rows)
day_ahead["period"] = pd.to_datetime(day_ahead["period"], utc=True)
day_ahead["value"] = pd.to_numeric(day_ahead["value"], errors="coerce")

day_ahead = (
    day_ahead[["period", "value"]]
    .rename(columns={"period": "timestamp", "value": "day_ahead_demand_forecast"})
    .set_index("timestamp")
    .sort_index()
)

model_data = co2.join(day_ahead, how="inner")

print("Joined rows:", len(model_data))
print("Date range:", model_data.index.min(), "to", model_data.index.max())
display(model_data.head())

Joined rows: 68186
Date range: 2019-01-01 00:00:00+00:00 to 2026-10-27 04:00:00+00:00


,carbon_intensity,day_ahead_demand_forecast
timestamp,,
2019-01-01 00:00:00+00:00,0.486023,22491
2019-01-01 01:00:00+00:00,0.563027,24210
2019-01-01 02:00:00+00:00,0.557796,27981
2019-01-01 03:00:00+00:00,0.543169,28321
2019-01-01 04:00:00+00:00,0.545020,27377


In [26]:
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error

y = model_data["carbon_intensity"].astype(float).sort_index()
demand_forecast = pd.to_numeric(
    model_data["day_ahead_demand_forecast"], errors="coerce"
).sort_index()

# Reindex to actual UTC hours so shifts mean hours, including across gaps.
hourly_index = pd.date_range(y.index.min(), y.index.max(), freq="h")
y = y.reindex(hourly_index)
demand_forecast = demand_forecast.reindex(hourly_index)

frame = pd.DataFrame(index=hourly_index)

# Past carbon-intensity inputs.
frame["lag_72h"] = y.shift(72)
frame["lag_96h"] = y.shift(96)
frame["lag_240h"] = y.shift(240)

# Predict intensity 24 hours ahead.
frame["target"] = y.shift(-24)

# Demand forecast for the hour being predicted.
frame["day_ahead_demand_forecast"] = demand_forecast.shift(-24)

# Same target hour one week earlier.
frame["baseline"] = y.shift(144)

features = [
    "lag_72h",
    "lag_96h",
    "lag_240h",
    "day_ahead_demand_forecast",
]

# Keep training labels before the test period.
cutoff = hourly_index[int(len(hourly_index) * 0.8)]
train = frame.loc[frame.index < cutoff - pd.Timedelta(hours=24)]
test = frame.loc[frame.index >= cutoff]

train = train.dropna(subset=features + ["target"])
test = test.dropna(subset=features + ["target", "baseline"])

model = HistGradientBoostingRegressor(
    max_iter=150,
    learning_rate=0.08,
    max_leaf_nodes=15,
    l2_regularization=1.0,
    random_state=42,
)
model.fit(train[features], train["target"])

model_mae = mean_absolute_error(test["target"], model.predict(test[features]))
baseline_mae = mean_absolute_error(test["target"], test["baseline"])

print(f"Training rows: {len(train):,}")
print(f"Test rows: {len(test):,}")
print(f"24-hour baseline MAE: {baseline_mae:.6f}")
print(f"24-hour model MAE:    {model_mae:.6f}")

Training rows: 53,849
Test rows: 13,075
24-hour baseline MAE: 0.068157
24-hour model MAE:    0.067897


In [27]:
import pandas as pd

def recommend_window(
    predicted_intensity: pd.Series,
    job_hours: int,
    max_wait_hours: int,
) -> pd.DataFrame:
    if job_hours <= 0:
        raise ValueError("job_hours must be greater than 0.")
    if max_wait_hours < 0:
        raise ValueError("max_wait_hours cannot be negative.")

    forecast = predicted_intensity.astype(float).copy()
    forecast.index = pd.to_datetime(forecast.index, utc=True)
    forecast = forecast.sort_index()

    first_hour = forecast.index[0]
    options = []

    # Try starting now, then each hour up to the user's wait limit.
    for wait in range(max_wait_hours + 1):
        window_start = first_hour + pd.Timedelta(hours=wait)
        window_hours = pd.date_range(
            start=window_start,
            periods=job_hours,
            freq="h",
        )
        values = forecast.reindex(window_hours)

        # Skip windows where any forecast hour is missing.
        if values.notna().all():
            options.append({
                "wait_hours": wait,
                "start": window_start,
                "end": window_hours[-1],
                "average_intensity": values.mean(),
            })

    if not options:
        raise ValueError(
            "The forecast does not cover any complete window for this job."
        )

    results = pd.DataFrame(options)
    run_now = results.loc[
        results["wait_hours"] == 0, "average_intensity"
    ]

    if run_now.empty:
        raise ValueError("Forecast does not cover a complete run-now window.")

    run_now_intensity = run_now.iloc[0]
    if run_now_intensity != 0:
        results["percent_saved_vs_run_now"] = (
            (run_now_intensity - results["average_intensity"])
            / run_now_intensity
            * 100
        )
    else:
        results["percent_saved_vs_run_now"] = 0.0

    return results.sort_values("average_intensity").reset_index(drop=True)


recommendations = recommend_window(
    predicted_intensity=carbon_forecast,
    job_hours=user_job_hours,
    max_wait_hours=user_max_wait_hours,
)

display(recommendations.head(3))

NameError: name 'carbon_forecast' is not defined

In [28]:
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor

# Set these to the user's preferences
user_job_hours = 4
user_max_wait_hours = 24

# How far ahead we need predictions:
# 24 hours of possible waiting + 4 hours to run the job = 28 hours
forecast_hours = user_max_wait_hours + user_job_hours

y = co2["carbon_intensity"].astype(float).sort_index()

# Ensure each row corresponds to an actual UTC hour
hourly_index = pd.date_range(y.index.min(), y.index.max(), freq="h")
y = y.reindex(hourly_index)

# Training data: for each historical forecast start, predict each future lead
training_parts = []

for lead in range(1, forecast_hours + 1):
    part = pd.DataFrame(index=y.index)
    part["lag_72h"] = y.shift(72)
    part["lag_96h"] = y.shift(96)
    part["lag_240h"] = y.shift(240)
    part["lead_hours"] = lead
    part["target"] = y.shift(-lead)
    training_parts.append(part)

training_data = pd.concat(training_parts).dropna()

features = ["lag_72h", "lag_96h", "lag_240h", "lead_hours"]

model = HistGradientBoostingRegressor(
    max_iter=150,
    learning_rate=0.08,
    max_leaf_nodes=15,
    l2_regularization=1.0,
    random_state=42,
)
model.fit(training_data[features], training_data["target"])

# Predict each hour after the latest observed hour
forecast_start = y.index.max()
latest_values = {
    "lag_72h": y.loc[forecast_start - pd.Timedelta(hours=72)],
    "lag_96h": y.loc[forecast_start - pd.Timedelta(hours=96)],
    "lag_240h": y.loc[forecast_start - pd.Timedelta(hours=240)],
}

future_rows = pd.DataFrame([
    {**latest_values, "lead_hours": lead}
    for lead in range(1, forecast_hours + 1)
])

future_index = pd.date_range(
    start=forecast_start + pd.Timedelta(hours=1),
    periods=forecast_hours,
    freq="h",
)

carbon_forecast = pd.Series(
    model.predict(future_rows[features]),
    index=future_index,
    name="predicted_carbon_intensity",
)

print(carbon_forecast)

2026-10-27 05:00:00+00:00    0.452339
2026-10-27 06:00:00+00:00    0.449489
2026-10-27 07:00:00+00:00    0.448654
2026-10-27 08:00:00+00:00    0.449545
2026-10-27 09:00:00+00:00    0.451695
2026-10-27 10:00:00+00:00    0.453903
2026-10-27 11:00:00+00:00    0.453488
2026-10-27 12:00:00+00:00    0.449855
2026-10-27 13:00:00+00:00    0.447866
2026-10-27 14:00:00+00:00    0.446961
2026-10-27 15:00:00+00:00    0.446961
2026-10-27 16:00:00+00:00    0.446961
2026-10-27 17:00:00+00:00    0.446961
2026-10-27 18:00:00+00:00    0.446961
2026-10-27 19:00:00+00:00    0.448035
2026-10-27 20:00:00+00:00    0.450257
2026-10-27 21:00:00+00:00    0.450332
2026-10-27 22:00:00+00:00    0.449358
2026-10-27 23:00:00+00:00    0.449358
2026-10-28 00:00:00+00:00    0.447745
2026-10-28 01:00:00+00:00    0.447745
2026-10-28 02:00:00+00:00    0.451712
2026-10-28 03:00:00+00:00    0.456400
2026-10-28 04:00:00+00:00    0.456400
2026-10-28 05:00:00+00:00    0.456400
2026-10-28 06:00:00+00:00    0.453247
2026-10-28 0

In [ ]:
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error

# model_data is the joined table with carbon_intensity and
# day_ahead_demand_forecast, indexed by UTC timestamp.
y = model_data["carbon_intensity"].astype(float).sort_index()
demand = model_data["day_ahead_demand_forecast"].astype(float).reindex(y.index)

# Make sure each index position represents one UTC hour.
hourly_index = pd.date_range(y.index.min(), y.index.max(), freq="h")
y = y.reindex(hourly_index)
demand = demand.reindex(hourly_index)

# Predict each hour needed for the user's scheduling window.
user_job_hours = 4
user_max_wait_hours = 24
forecast_hours = user_job_hours + user_max_wait_hours

# Make one training example for every historical start time and forecast lead.
parts = []

for lead in range(1, forecast_hours + 1):
    part = pd.DataFrame(index=y.index)
    part["origin"] = y.index
    part["lead_hours"] = lead
    part["lag_72h"] = y.shift(72)
    part["lag_96h"] = y.shift(96)
    part["lag_240h"] = y.shift(240)

    # Demand forecast for the hour we're trying to predict
    part["target_demand_forecast"] = demand.shift(-lead)

    # Carbon intensity for that same future hour
    part["target"] = y.shift(-lead)

    # Same target hour one week earlier, used as a simple benchmark
    part["baseline"] = y.shift(168 - lead)

    parts.append(part)

examples = pd.concat(parts).dropna()

# Split by time, keeping later examples for evaluation.
cutoff = y.index[int(len(y) * 0.8)]
train = examples[examples["origin"] < cutoff - pd.Timedelta(hours=forecast_hours)]
test = examples[examples["origin"] >= cutoff]

lag_features = ["lag_72h", "lag_96h", "lag_240h", "lead_hours"]
demand_features = lag_features + ["target_demand_forecast"]

# Use only rows with demand available so both models get the same test examples.
train = train.dropna(subset=demand_features + ["target", "baseline"])
test = test.dropna(subset=demand_features + ["target", "baseline"])

settings = dict(
    max_iter=150,
    learning_rate=0.08,
    max_leaf_nodes=15,
    l2_regularization=1.0,
    random_state=42,
)

lag_model = HistGradientBoostingRegressor(**settings)
demand_model = HistGradientBoostingRegressor(**settings)

lag_model.fit(train[lag_features], train["target"])
demand_model.fit(train[demand_features], train["target"])

baseline_mae = mean_absolute_error(test["target"], test["baseline"])
lag_mae = mean_absolute_error(test["target"], lag_model.predict(test[lag_features]))
demand_mae = mean_absolute_error(
    test["target"],
    demand_model.predict(test[demand_features]),
)

print(f"Training examples:       {len(train):,}")
print(f"Test examples:           {len(test):,}")
print(f"Same-hour-last-week MAE: {baseline_mae:.6f}")
print(f"Lag-only model MAE:      {lag_mae:.6f}")
print(f"Lag + demand model MAE:  {demand_mae:.6f}")

Training examples:       1,503,757
Test examples:           365,837
Same-hour-last-week MAE: 0.068265
Lag-only model MAE:      0.070482
Lag + demand model MAE:  0.071830


In [31]:
import pandas as pd
import requests

weather_points = {
    "sacramento": (38.58, -121.49),
    "fresno": (36.74, -119.79),
    "san_diego": (32.72, -117.16),
}

weather_variables = (
    "temperature_2m,wind_speed_10m,cloud_cover,"
    "shortwave_radiation,precipitation"
)

start_date = max(co2.index.min().strftime("%Y-%m-%d"), "2022-01-01")
end_date = co2.index.max().strftime("%Y-%m-%d")

weather_frames = []

for place, (latitude, longitude) in weather_points.items():
    response = requests.get(
        "https://archive-api.open-meteo.com/v1/archive",
        params={
            "latitude": latitude,
            "longitude": longitude,
            "start_date": start_date,
            "end_date": end_date,
            "hourly": weather_variables,
            "timezone": "UTC",
        },
        timeout=60,
    )
    response.raise_for_status()

    hourly = response.json()["hourly"]
    part = pd.DataFrame(hourly)
    part["timestamp"] = pd.to_datetime(part.pop("time"), utc=True)
    part = part.set_index("timestamp").add_prefix(f"{place}_")
    weather_frames.append(part)

weather = pd.concat(weather_frames, axis=1).sort_index()

print("Weather rows:", len(weather))
print("Weather columns:", weather.columns.tolist())
display(weather.head())

Weather rows: 42264
Weather columns: ['sacramento_temperature_2m', 'sacramento_wind_speed_10m', 'sacramento_cloud_cover', 'sacramento_shortwave_radiation', 'sacramento_precipitation', 'fresno_temperature_2m', 'fresno_wind_speed_10m', 'fresno_cloud_cover', 'fresno_shortwave_radiation', 'fresno_precipitation', 'san_diego_temperature_2m', 'san_diego_wind_speed_10m', 'san_diego_cloud_cover', 'san_diego_shortwave_radiation', 'san_diego_precipitation']


,sacramento_temperature_2m,sacramento_wind_speed_10m,sacramento_cloud_cover,sacramento_shortwave_radiation,sacramento_precipitation,fresno_temperature_2m,fresno_wind_speed_10m,fresno_cloud_cover,fresno_shortwave_radiation,fresno_precipitation,san_diego_temperature_2m,san_diego_wind_speed_10m,san_diego_cloud_cover,san_diego_shortwave_radiation,san_diego_precipitation
timestamp,,,,,,,,,,,,,,,
2022-01-01 00:00:00+00:00,10.0,8.4,1,176.0,0.0,9.7,15.9,96,56.0,0.0,13.3,23.2,39,186.0,0.0
2022-01-01 01:00:00+00:00,6.2,9.2,0,36.0,0.0,8.6,14.5,78,22.0,0.0,13.0,19.4,100,17.0,0.1
2022-01-01 02:00:00+00:00,4.9,9.5,0,0.0,0.0,7.3,11.0,65,0.0,0.0,12.7,17.7,100,0.0,0.0
2022-01-01 03:00:00+00:00,3.8,10.6,0,0.0,0.0,6.9,10.7,36,0.0,0.0,12.5,15.1,100,0.0,0.0
2022-01-01 04:00:00+00:00,3.1,10.3,0,0.0,0.0,5.8,10.4,11,0.0,0.0,12.5,13.0,100,0.0,0.0


In [36]:
import pandas as pd
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error

# Use only dates where carbon, demand, and weather overlap.
data_start = max(model_data.index.min(), weather.index.min())
data_end = min(model_data.index.max(), weather.index.max())

hourly_index = pd.date_range(
    start=data_start.ceil("h"),
    end=data_end.floor("h"),
    freq="h",
)

y = pd.to_numeric(
    model_data["carbon_intensity"], errors="coerce"
).reindex(hourly_index)

demand = pd.to_numeric(
    model_data["day_ahead_demand_forecast"], errors="coerce"
).reindex(hourly_index)

weather_hourly = weather.apply(
    pd.to_numeric, errors="coerce"
).reindex(hourly_index)

# At forecast start t, use weather observed at t - 6 hours.
weather_lagged = weather_hourly.shift(6).add_suffix("_lag6h")
weather_features = list(weather_lagged.columns)

# Sample forecast starts every 6 hours to keep this first XGBoost run manageable.
origins = hourly_index[::6]
forecast_hours = user_job_hours + user_max_wait_hours

parts = []

for lead in range(1, forecast_hours + 1):
    part = pd.DataFrame(index=origins)
    part["origin"] = origins
    part["lead_hours"] = lead

    # Carbon readings known before the forecast start.
    for lag in [72, 96, 240]:
        lookup_times = origins - pd.Timedelta(hours=lag)
        part[f"carbon_lag_{lag}h"] = y.reindex(lookup_times).to_numpy()

    # Weather readings from six hours before the forecast start.
    part[weather_features] = weather_lagged.reindex(origins).to_numpy()

    # EIA day-ahead demand forecast for the hour being predicted.
    target_times = origins + pd.Timedelta(hours=lead)
    part["target_demand_forecast"] = demand.reindex(target_times).to_numpy()

    # Actual future carbon intensity: the answer used for training/scoring.
    part["target"] = y.reindex(target_times).to_numpy()

    # Same target hour last week, used both as a model input and benchmark.
    part["baseline"] = y.reindex(
        target_times - pd.Timedelta(hours=168)
    ).to_numpy()

    parts.append(part)

examples = pd.concat(parts).dropna()

# Keep later dates separate so evaluation uses data the model did not train on.
cutoff = hourly_index[int(len(hourly_index) * 0.8)]
train = examples[
    examples["origin"] < cutoff - pd.Timedelta(hours=forecast_hours)
]
test = examples[examples["origin"] >= cutoff]

features = (
    ["carbon_lag_72h", "carbon_lag_96h", "carbon_lag_240h",
     "lead_hours", "target_demand_forecast", "baseline"]
    + weather_features
)

model = XGBRegressor(
    n_estimators=400,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=2.0,
    objective="reg:squarederror",
    tree_method="hist",
    n_jobs=-1,
    random_state=42,
)

model.fit(train[features], train["target"])

xgb_mae = mean_absolute_error(
    test["target"],
    model.predict(test[features]),
)
baseline_mae = mean_absolute_error(
    test["target"],
    test["baseline"],
)

print(f"Training examples: {len(train):,}")
print(f"Test examples:     {len(test):,}")
print(f"Same-hour-last-week MAE: {baseline_mae:.6f}")
print(f"Weather + demand XGBoost MAE: {xgb_mae:.6f}")

Training examples: 151,835
Test examples:     37,434
Same-hour-last-week MAE: 0.071676
Weather + demand XGBoost MAE: 0.091935


In [ ]:
bq7pMrCjgkbr96BwWqBj9gbPcpWW9rLrNmhjnBhz